# End-to-End Sound Generation on Real Audio Data (ALD-SC)

This notebook trains the full ALD-SC pipeline end-to-end on the audio files in `data/` (NSynth-style .wav files). It uses the real frozen `EnCodec` encoder and the graph-structured decoder.

## Pipeline
1. Load .wav files from `data/` using `AudioFolderDataset`.
2. Extract real EnCodec latents and build the frozen ArrowSpace prior.
3. Train graph decoder vs. matched-capacity baseline decoder.
4. Train a 1-D DiT denoiser on the real EnCodec latent space.
5. Sample, decode, and evaluate.

In [ ]:
import os
from pathlib import Path

# --- Generation knobs ---
SEED = 3407
STEPS = 50
TEMPERATURE = 0.85
USE_C_SPEC = True

# --- Dataset knobs ---
DATA_DIR = Path(os.getcwd()).parent / 'data'           # directory containing .wav files (NSynth-style)
AUDIO_LENGTH = 96000        # 4 seconds @ 24kHz (NSynth clips are 4s)
SAMPLE_RATE = 24000
SUBSET_SIZE = 256           # set to None to use all files

# --- Model knobs ---
Q = 8                       # prior chart dimension
K = 4                       # prior knn
BASE_CHANNELS = 32

# --- Training knobs ---
DECODER_EPOCHS = 20
DIFFUSION_EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3

LATENT_LENGTH = AUDIO_LENGTH // 320  # EnCodec 24kHz stride
print(f'Data dir: {DATA_DIR}')
print(f'Audio: {AUDIO_LENGTH/SAMPLE_RATE:.1f}s ({AUDIO_LENGTH} samples) -> latent length {LATENT_LENGTH}')
print(f'Subset size: {SUBSET_SIZE}')
print(f'Knobs: seed={SEED}, steps={STEPS}, temp={TEMPERATURE}, use_c_spec={USE_C_SPEC}')

## Imports

In [ ]:
import random
from pathlib import Path

import torch
import torch.nn as nn
import torchaudio
from IPython.display import Audio, display

from ald_sc.build_prior import build_arrow_prior
from ald_sc.audio_codec import EnCodecEncoder, AudioVAE, BaselineAudioDecoder
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.dit import MinimalDiT
from ald_sc.data import AudioFolderDataset, build_audio_dataloader
from ald_sc.losses import ALDSCLoss
from ald_sc.schedule import CosineSchedule
from ald_sc.sampling import sample_ddim
from ald_sc.trainer import train_audio_decoder, train_audio_diffusion, log_training

device = torch.device('cpu')
random.seed(SEED)
torch.manual_seed(SEED)
print('Imports done. Device:', device)
print('EnCodecEncoder loaded lazily — first encode call may download weights.')

## Step 1: Load Real Audio Dataset

In [ ]:
# List all .wav files in the data directory
all_files = sorted(Path(DATA_DIR).glob('*.wav'))
print(f'Found {len(all_files)} audio files in {DATA_DIR}')

# Optional reproducible subset for faster demo runs
if SUBSET_SIZE is not None and len(all_files) > SUBSET_SIZE:
    random.seed(SEED)
    selected_files = random.sample(all_files, SUBSET_SIZE)
else:
    selected_files = all_files

print(f'Using {len(selected_files)} files for training')

# AudioFolderDataset loads, resamples to 24kHz, and crops/pads to AUDIO_LENGTH
dataset = AudioFolderDataset(
    root=DATA_DIR,
    audio_length=AUDIO_LENGTH,
    sample_rate=SAMPLE_RATE,
)

# Override file list to use the subset
dataset.files = selected_files
print(f'Dataset: {len(dataset)} clips, shape {dataset[0].shape}')

## Step 2: Build the ArrowSpace Prior from Real EnCodec Features

In [ ]:
# Real frozen EnCodec encoder
encoder = EnCodecEncoder(sample_rate=SAMPLE_RATE, bandwidth=24)

loader = build_audio_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# Extract pooled EnCodec features for the prior
features = []
for batch in loader:
    z = encoder.extract_features(batch)
    features.append(z.mean(dim=2))
embeddings = torch.cat(features, dim=0)
print(f'Corpus EnCodec embeddings: {embeddings.shape}')

# Build the frozen ArrowSpace prior
prior = build_arrow_prior(embeddings, q=Q, k=K)
print(f'L_F: {prior.L_F.shape}, U_q: {prior.U_q.shape}, q={prior.q}')

## Step 3: Train Graph Decoder vs. Baseline Decoder

In [ ]:
# Graph decoder (uses real EnCodec latents + ArrowSpace prior)
graph_decoder = GraphDecoder(
    latent_channels=128,
    out_channels=1,
    feature_dim=128,
    base_channels=BASE_CHANNELS,
    prior=prior,
    upsample_strides=(2, 4, 5, 8),
)

# Matched-capacity baseline decoder (no graph structure)
baseline_decoder = BaselineAudioDecoder(
    latent_channels=128,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    upsample_strides=(2, 4, 5, 8),
)

loss_fn = ALDSCLoss(
    prior=prior,
    lambda_rec=1.0,
    lambda_stft=0.0,
    lambda_chart=0.5,
    lambda_smooth=0.1,
)
train_loader = build_audio_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Train graph decoder
graph_vae = AudioVAE(encoder=encoder, decoder=graph_decoder)
print('Training graph decoder on real EnCodec latents...')
graph_losses = list(train_audio_decoder(
    train_loader, graph_vae, prior, loss_fn,
    epochs=DECODER_EPOCHS, lr=LR, device=device,
))
print(f'  Loss: {graph_losses[0]["loss"]:.4f} -> {graph_losses[-1]["loss"]:.4f}')

# Train baseline decoder
baseline_vae = AudioVAE(encoder=encoder, decoder=baseline_decoder)
print('Training baseline decoder...')
baseline_losses = list(train_audio_decoder(
    train_loader, baseline_vae, prior, loss_fn,
    epochs=DECODER_EPOCHS, lr=LR, device=device,
))
print(f'  Loss: {baseline_losses[0]["loss"]:.4f} -> {baseline_losses[-1]["loss"]:.4f}')

## Step 4: Train the 1-D DiT Denoiser

In [ ]:
dit = MinimalDiT(
    latent_channels=128,
    latent_length=LATENT_LENGTH,
    patch_size=8,
    dim=64,
    depth=2,
    num_heads=4,
    spec_dim=3 * Q,
)
sched = CosineSchedule(num_steps=1000)

# Freeze VAE for diffusion training
for p in graph_vae.parameters():
    p.requires_grad_(False)

print(f'Training 1-D DiT on real EnCodec latents (latent_length={LATENT_LENGTH})...')
diff_losses = list(train_audio_diffusion(
    train_loader, graph_vae, dit, prior, sched,
    epochs=DIFFUSION_EPOCHS, lr=LR, device=device,
))
print(f'  Loss: {diff_losses[0]["loss"]:.4f} -> {diff_losses[-1]["loss"]:.4f}')

## Step 5: Generate Sound

In [ ]:
# Sample latent z from noise
torch.manual_seed(SEED)
dit = dit.eval()
z = sample_ddim(dit, sched, batch_size=1, steps=STEPS, seed=SEED, device=device)

# Apply temperature scaling
z = z * TEMPERATURE
print(f'Sampled z: {z.shape}')

# Derive c_spec from z (self-consistent decoding)
a = z.mean(dim=2)
c_spec = prior.chart_energy_descriptor(a)

# Decode with graph decoder
with torch.no_grad():
    if USE_C_SPEC:
        audio_graph = graph_decoder(z, c_spec)
    else:
        audio_graph = graph_decoder(z, torch.zeros_like(c_spec))
    audio_baseline = baseline_decoder(z)

# Normalize for playback
def normalize(audio):
    audio = audio.squeeze(0)
    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak
    return audio

audio_graph_norm = normalize(audio_graph)
audio_baseline_norm = normalize(audio_baseline)

print(f'Graph decoder audio: {audio_graph_norm.shape}, {audio_graph_norm.shape[-1]/SAMPLE_RATE:.2f}s')
print(f'Baseline decoder audio: {audio_baseline_norm.shape}, {audio_baseline_norm.shape[-1]/SAMPLE_RATE:.2f}s')

In [ ]:
print('Graph decoder output:')
display(Audio(audio_graph_norm.numpy(), rate=SAMPLE_RATE))

In [ ]:
print('Baseline decoder output:')
display(Audio(audio_baseline_norm.numpy(), rate=SAMPLE_RATE))

In [ ]:
print('Real training clip (reference):')
real_clip = dataset[0]  # (1, T)
display(Audio(real_clip.numpy(), rate=SAMPLE_RATE))

## Step 6: Evaluation

In [ ]:
eval_loader = build_audio_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=False)

def eval_reconstruction(vae, loader, loss_fn, device):
    vae.eval()
    total_rec, total_chart, n = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            x = batch.to(device)
            z, A, c_spec, x_hat = vae(x, prior)
            losses = loss_fn(x, x_hat, A, A.detach())
            total_rec += losses['rec'].item()
            total_chart += losses['chart'].item()
            n += 1
    return {'rec': total_rec/n, 'chart': total_chart/n}

graph_metrics = eval_reconstruction(graph_vae, eval_loader, loss_fn, device)
baseline_metrics = eval_reconstruction(baseline_vae, eval_loader, loss_fn, device)

print('=== Reconstruction Comparison (real EnCodec latents) ===')
print(f'Graph decoder:    L1={graph_metrics["rec"]:.6f}  chart={graph_metrics["chart"]:.6f}')
print(f'Baseline decoder: L1={baseline_metrics["rec"]:.6f}  chart={baseline_metrics["chart"]:.6f}')
diff = baseline_metrics['rec'] - graph_metrics['rec']
print(f'Graph improvement: {diff:+.6f} (positive = graph is better)')

## Step 7: lambda_ED Ablation

In [ ]:
class NoCSPecVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, prior):
        z, a, c_spec = self.encoder.encode(x, prior)
        zero_cspec = torch.zeros_like(c_spec)
        x_hat = self.decoder(z, zero_cspec)
        return z, a, c_spec, x_hat

ablation_vae = NoCSPecVAE(encoder, graph_decoder)
ablation_metrics = eval_reconstruction(ablation_vae, eval_loader, loss_fn, device)

print('=== lambda_ED Ablation ===')
print(f'With c_spec:    L1={graph_metrics["rec"]:.6f}  chart={graph_metrics["chart"]:.6f}')
print(f'Without c_spec: L1={ablation_metrics["rec"]:.6f}  chart={ablation_metrics["chart"]:.6f}')
diff = ablation_metrics['rec'] - graph_metrics['rec']
print(f'lambda_ED effect: {diff:+.6f} (positive = gating helps)')

## Summary

In [ ]:
print('=== Summary ===')
print(f'Encoder: real EnCodec 24kHz (frozen)')
print(f'Dataset: {DATA_DIR} ({len(dataset)} clips, {AUDIO_LENGTH/SAMPLE_RATE:.1f}s each)')
print(f'Prior: ArrowSpace q={Q}, k={K}')
print(f'Graph decoder loss: {graph_losses[-1]["loss"]:.4f}')
print(f'Baseline decoder loss: {baseline_losses[-1]["loss"]:.4f}')
print(f'DiT loss: {diff_losses[-1]["loss"]:.4f}')
print('\nRe-run the knob cell with different SUBSET_SIZE, SEED, STEPS,')
print('TEMPERATURE, or USE_C_SPEC to explore the generated sound space.')